<a href="https://colab.research.google.com/github/WeiDeHuang1019/eyes-catching/blob/main/Eyes_Catching_HW1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Eyes-Catching 嵌入式影像處理HW1[F114112124黃維得]

### (1)find ROI part

In [ ]:
import numpy as np
import cv2
from google.colab.patches import cv2_imshow

#讀取圖片
im = cv2.imread("face.jpg")
im_drawCircle = im.copy()
#im = cv2.convertScaleAbs(im, alpha=2, beta=0)
cv2_imshow(im)

#取得圖片寬
img_width=im.shape[1]
#dot = int(img_width/10)
print("Image Width =", img_width)

#轉灰階
img_gray = cv2.cvtColor(im, cv2.COLOR_BGR2GRAY)
cv2_imshow(img_gray)

#二質化
A, th = cv2.threshold(img_gray, 215, 255, cv2.THRESH_BINARY_INV)
cv2_imshow(th)

#逐列掃描分析輪廓分布 ->找臉中心
H, W = th.shape
x_centers = []
rows = []
for y in range(H):
    xs = np.where(th[y] > 0)[0]  # 找這一列所有「人」的 pixel
    if len(xs) < 10:  # 太少代表不是有效列（例如雜訊）
        continue
    x_left = xs[0]
    x_right = xs[-1]
    width = x_right - x_left
    x_mid = (x_left + x_right) / 2.0
    x_centers.append(x_mid)
    rows.append((y, x_left, x_right, width, x_mid))
x_face = int(np.median(x_centers)) #臉中心(x軸)
print("Face center x =", x_face)

# 抓頭頂位置
y_top = rows[0][0] #頭頂位置(y軸)
print("y_top =", y_top)
ys = np.array([r[0] for r in rows])
widths = np.array([r[3] for r in rows])

# 抓脖子位置
y_min_search = int(H * 0.15)
y_max_search = int(H * 0.65)
candidate = [(y, w) for (y, _, _, w, _) in rows if y_min_search <= y <= y_max_search]
y_vals = np.array([c[0] for c in candidate])
w_vals = np.array([c[1] for c in candidate])
idx = np.argmin(w_vals)
y_neck = int(y_vals[idx]) #脖子位置(y軸)
print("y_neck =", y_neck)

# 定義單位長度
dot = int((y_neck - y_top)/10)
print("dot =", dot)

# 抓眼睛位置
head_h = y_neck - y_top
y_eye_center = int(y_top + 0.55 * head_h) #眼睛位置(y軸)
y1 = int(y_top + 0.25 * head_h)
y2 = int(y_top + 0.50 * head_h)
print("eye center y =", y_eye_center)
print("eye ROI y-range =", y1, y2)

# 以臉與眼睛中心，畫 ROI 範圍
x1 = max(x_face - int(dot*3), 0)
y1 = max(y_eye_center - int(dot*1.2), 0)
x2 = min(x_face + int(dot*3), im.shape[1])
y2 = min(y_eye_center + int(dot*1.2), im.shape[0])
cv2.rectangle(im, (x1, y1), (x2, y2), (255, 0, 0), 2)
roi = im[y1:y2, x1:x2].copy()
cv2_imshow(im)


###(2)find eyeball part

In [ ]:
im_drawCircle = im.copy()
cv2_imshow(roi)
roi_gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
cv2_imshow(roi_gray)
roi_gray = cv2.GaussianBlur(roi_gray, (9, 9), 0)
roi_gray = cv2.GaussianBlur(roi_gray, (9, 9), 0)
cv2_imshow(roi_gray)
#_, th_in_roi = cv2.threshold(roi_gray, 35, 255, cv2.THRESH_BINARY_INV)
th_in_roi = cv2.adaptiveThreshold(
    roi_gray,
    255,
    cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
    cv2.THRESH_BINARY_INV,
    21,   # block size，必須奇數
    7     # C，越大越不容易變白
)

cv2_imshow(th_in_roi)
roi_Canny = cv2.Canny(th_in_roi, 15, 100)
cv2_imshow(roi_Canny)
#th_in_roi = cv2.morphologyEx(th_in_roi, cv2.MORPH_DILATE, np.ones((int(dot/20), int(dot/20)), np.uint8))

#用霍夫圓抓眼珠
circles_1 = cv2.HoughCircles(roi_Canny, cv2.HOUGH_GRADIENT, dp=1.2,
               minDist=dot, param1=150, param2=50, minRadius=int(dot/10), maxRadius=int(dot/2))
if circles_1 is not None:
    circles_1 = np.uint16(np.around(circles_1))
    for i in circles_1[0, :]:
        x, y, r = i
        cv2.circle(im_drawCircle, (x+x1, y+y1), r, (0, 255, 0), 2)
        cv2.circle(im_drawCircle, (x+x1, y+y1), 2, (0, 0, 255), 3)

    print(f"第一階段偵測到 {len(circles_1[0])} 個圓")
else:
    print("第一階段未偵測到任何圓形")

roi_drawCircle = im_drawCircle[y1:y2, x1:x2].copy()
cv2_imshow(roi_drawCircle)
cv2_imshow(im_drawCircle)